# 04 · DCF and reverse DCF
Runs for the active pitch (**UNH**), with tables shown for it. It also redraws the figures for COST (comparison) and V (archived first pick), using the same code and assumptions. Figures go to `figures/<TICKER>/`.

Two-stage FCF DCF. WACC inputs and their sources are listed below. The terminal growth and fade length are stated assumptions (`analysis.ASSUMPTIONS`) and are the same for every ticker. The reverse DCF solves for the stage-1 growth that the as-of price implies.

In [1]:
import warnings

warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt

from pitch import analysis as a
from pitch.plotting import apply_theme, save
from pitch.universe import ACTIVE

apply_theme()
TICKERS = [ACTIVE, "COST", "V"]  # active pitch, comparison, archived first pick
a.set_pitch(ACTIVE)

In [2]:
a.dcf()

{'base_fcf': 16075000000.0,
 'base_fcf_source': 'FY2025 operating cash flow - capex (SEC 10-K)',
 'stage1_growth': 0.1372,
 'stage1_growth_source': 'median rolling 5y FCF CAGR, FY2010 onward',
 'wacc_inputs': {'risk_free': {'value': 0.0529,
   'source': 'FRED DGS10 on 2026-09-30'},
  'beta': {'value': 0.4067,
   'source': '5y daily stock vs SPY returns to as-of (OLS slope)'},
  'equity_risk_premium': {'value': 0.0835,
   'source': 'Ken French Mkt-RF, arithmetic mean 1926-07 to 202608, x12'},
  'cost_of_equity': {'value': 0.0868, 'source': 'CAPM: rf + beta x ERP'},
  'pre_tax_cost_of_debt': {'value': 0.0553,
   'source': 'FY2025 interest expense / average total debt (SEC 10-K)'},
  'tax_rate': {'value': 0.1286,
   'source': 'FY2025 income tax / pretax income (SEC 10-K)'},
  'equity_weight': {'value': 0.82,
   'source': 'market cap (as-of price x Yahoo shares) / (market cap + book debt)'},
  'wacc': {'value': 0.0799, 'source': 'we x ke + (1 - we) x kd x (1 - t)'}},
 'price': 367.08,
 'im

In [3]:
import numpy as np

gs = np.linspace(-0.05, 0.4, 46)
for t in TICKERS:
    a.set_pitch(t)
    d = a.dcf()
    w = d["wacc_inputs"]["wacc"]["value"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(gs, [a.dcf_value(g, w) for g in gs])
    ax.axhline(d["price"], color="#f59e0b", ls="--", label=f"as-of price {d['price']}")
    ax.axvline(d["stage1_growth"], color="#94a3b8", ls=":", label="base stage-1 growth")
    ax.axvline(d["reverse_dcf_stage1_growth"], color="#10B981", ls=":", label="price-implied growth")
    ax.set_title(t)
    ax.set_xlabel("stage-1 FCF growth")
    ax.set_ylabel("value per share")
    ax.legend(frameon=False)
    save(fig, f"{t}/04_dcf_growth_sensitivity")
a.set_pitch(ACTIVE)

findfont: Failed to find font weight semibold for Inter, now using 400.
